In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

/Users/sanjaymahto/About-AI/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
from langchain_groq import ChatGroq
llm_model = "llama-3.1-8b-instant"
llm = ChatGroq(
    model=llm_model,
    temperature=0,
)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x112fc4ca0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112fcc340>, model_name='llama-3.1-8b-instant', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogState) -> BlogState: 
    # Fetcht the title 
    title = state['topic']
    
    # call the llm Generate outline 
    prompt = f"Generate a detailed outline for a blog post about {title}"
    outline = llm.invoke(prompt).content
    
    
    
    # Update state 
    state['outline'] = outline
    return state
    
    


def create_blog(state: BlogState) -> BlogState: 
    
    title = state['topic']
    outline = state['outline']
    
    # Create a new prompt 
    prompt = f"write a detailed blog on the title -{title} using the following outline: \n  {outline}"
    content = llm.invoke(prompt).content
    
    state['content'] = content
    return state




In [7]:
graph = StateGraph(BlogState)

# Nodes 
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# add edges 
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)


workflow = graph.compile()


In [8]:
inital_state = {
    "topic": "Write about raise of Rohit sharma."
}

final_state = workflow.invoke(inital_state)


In [9]:
final_state['outline']

"**Title:** The Rise of Rohit Sharma: A Journey to Becoming India's Cricketing Sensation\n\n**I. Introduction**\n\n- Brief overview of Rohit Sharma's early life and cricketing background\n- Importance of Rohit Sharma in Indian cricket\n- Thesis statement: Rohit Sharma's journey from a young cricketer to a world-class batsman is a testament to his hard work, dedication, and perseverance.\n\n**II. Early Life and Cricketing Background**\n\n- Rohit Sharma's birth and upbringing in Nagpur, Maharashtra\n- Introduction to cricket at a young age and his early cricketing experiences\n- Early successes and failures in domestic cricket\n\n**III. Breakthrough in Domestic Cricket**\n\n- Rohit Sharma's rise to prominence in the Indian Premier League (IPL)\n- His performances for the Mumbai Indians and other domestic teams\n- Key milestones and achievements in domestic cricket\n\n**IV. International Debut and Early Success**\n\n- Rohit Sharma's international debut in 2007\n- His early struggles and s

In [10]:
print(final_state['content'])

**The Rise of Rohit Sharma: A Journey to Becoming India's Cricketing Sensation**

Rohit Sharma, one of the most successful batsmen in Indian cricket history, has come a long way since his early days as a young cricketer from Nagpur, Maharashtra. With a career spanning over 15 years, Rohit has etched his name in the annals of Indian cricket, leaving behind a trail of records and achievements that are a testament to his hard work, dedication, and perseverance.

**I. Introduction**

Born on April 30, 1987, in Nagpur, Maharashtra, Rohit Sharma was introduced to cricket at a young age by his father, Rajesh Sharma, who was a cricket enthusiast himself. Rohit's early cricketing experiences were marked by his participation in local tournaments and his passion for the game only grew stronger with each passing day. As he grew older, Rohit's talent and dedication caught the attention of cricket scouts, and he soon found himself playing for the Mumbai Indians in the Indian Premier League (IPL).

R

In [15]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str
    eval: str

In [19]:
def create_outline(state: BlogState) -> BlogState: 
    # Fetcht the title 
    title = state['topic']
    
    # call the llm Generate outline 
    prompt = f"Generate a detailed outline for a blog post about {title}"
    outline = llm.invoke(prompt).content
    
    
    
    # Update state 
    state['outline'] = outline
    return state
    
    


def create_blog(state: BlogState) -> BlogState: 
    
    title = state['topic']
    outline = state['outline']
    
    # Create a new prompt 
    prompt = f"write a detailed blog on the title -{title} using the following outline: \n  {outline}"
    content = llm.invoke(prompt).content
    
    state['content'] = content
    return state

def evaluate_blog(state: BlogState) -> BlogState:
    content = state['content']
    
    prompt = f"Based on this outline {content}, rate the blog on a scale of 1 to 10 for quality and coherence."
    eval_score = llm.invoke(prompt).content

    state['eval'] = eval_score
    return state


In [20]:
graph = StateGraph(BlogState)

# Nodes 
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# add edges 
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog','evaluate_blog')
graph.add_edge('evaluate_blog', END)


workflow = graph.compile()


In [21]:
inital_state = {
    "topic": "Write about raise of Rohit sharma."
}

final_state = workflow.invoke(inital_state)


In [22]:
print(final_state)

{'topic': 'Write about raise of Rohit sharma.', 'outline': "**Title:** The Rise of Rohit Sharma: A Journey to Becoming India's Cricketing Sensation\n\n**I. Introduction**\n\n- Brief overview of Rohit Sharma's early life and cricketing background\n- Importance of Rohit Sharma in Indian cricket\n- Thesis statement: Rohit Sharma's journey from a young cricketer to a world-class batsman is a testament to his hard work, dedication, and perseverance.\n\n**II. Early Life and Cricketing Background**\n\n- Rohit Sharma's birth and upbringing in Nagpur, Maharashtra\n- Introduction to cricket at a young age and his early cricketing experiences\n- Early successes and failures in domestic cricket\n\n**III. Breakthrough in Domestic Cricket**\n\n- Rohit Sharma's rise to prominence in the Indian Premier League (IPL)\n- His performances for the Mumbai Indians and other domestic teams\n- Key milestones and achievements in domestic cricket\n\n**IV. International Debut and Early Success**\n\n- Rohit Sharma

In [23]:
final_state['eval']

'I would rate this blog post a 9 out of 10 for quality and coherence. Here\'s why:\n\n**Strengths:**\n\n1. **Clear structure**: The blog post follows a logical and easy-to-follow structure, with each section building on the previous one to tell a cohesive story.\n2. **Well-researched content**: The post is well-researched, with a good balance of Rohit Sharma\'s early life, cricketing background, breakthroughs, and achievements.\n3. **Engaging writing style**: The writing is engaging, with a conversational tone that makes the reader feel like they\'re reading a biography rather than a dry, factual account.\n4. **Use of examples and anecdotes**: The post uses examples and anecdotes to illustrate Rohit Sharma\'s journey, making the story more relatable and interesting.\n5. **Good use of transitions**: The post uses transitional phrases and sentences to connect each section, making the narrative flow smoothly.\n\n**Weaknesses:**\n\n1. **Some sections feel a bit repetitive**: While the post